# Zadanie 1 — PgVector: Nearest Neighbor

Wyszukiwanie najbliższego wektora (nearest neighbor) w PostgreSQL przy użyciu pgvector.

**Dane:** tabela `products_ex1` z wektorami 3-wymiarowymi  
**Operator:** `<=>` — odległość kosinusowa (cosine distance)

In [8]:
import psycopg2

conn = psycopg2.connect(
    host="pgvector_snoql_lab",
    database="vectordb",
    user="user",
    password="password"
)
cur = conn.cursor()
print("Połączono z bazą danych.")

Połączono z bazą danych.


## Tworzenie tabeli i wstawianie danych

In [9]:
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

cur.execute("""
    SELECT EXISTS (
        SELECT FROM information_schema.tables
        WHERE table_name = 'products_ex1'
    );
""")
table_exists = cur.fetchone()[0]

if table_exists:
    cur.execute("SELECT COUNT(*) FROM products_ex1;")
    count = cur.fetchone()[0]
    print(f"Tabela products_ex1 już istnieje ({count} wierszy) — pomijam tworzenie.")
else:
    cur.execute("""
        CREATE TABLE products_ex1 (
            id SERIAL PRIMARY KEY,
            name TEXT,
            embedding VECTOR(3)
        );
    """)
    conn.commit()
    print("Tabela products_ex1 utworzona.")

Tabela products_ex1 już istnieje (30 wierszy) — pomijam tworzenie.


In [10]:
data = [
    ('P1',  '[0.91, 0.05, 0.02]'),
    ('P2',  '[0.88, 0.10, 0.01]'),
    ('P3',  '[0.85, 0.15, 0.05]'),
    ('P4',  '[0.80, 0.20, 0.10]'),
    ('P5',  '[0.78, 0.22, 0.15]'),
    ('P6',  '[0.75, 0.25, 0.10]'),
    ('P7',  '[0.70, 0.30, 0.20]'),
    ('P8',  '[0.65, 0.35, 0.25]'),
    ('P9',  '[0.60, 0.40, 0.30]'),
    ('P10', '[0.55, 0.45, 0.35]'),
    ('P11', '[0.05, 0.90, 0.02]'),
    ('P12', '[0.10, 0.88, 0.01]'),
    ('P13', '[0.15, 0.85, 0.05]'),
    ('P14', '[0.20, 0.80, 0.10]'),
    ('P15', '[0.22, 0.78, 0.15]'),
    ('P16', '[0.25, 0.75, 0.10]'),
    ('P17', '[0.30, 0.70, 0.20]'),
    ('P18', '[0.35, 0.65, 0.25]'),
    ('P19', '[0.40, 0.60, 0.30]'),
    ('P20', '[0.45, 0.55, 0.35]'),
    ('P21', '[0.33, 0.33, 0.34]'),
    ('P22', '[0.30, 0.30, 0.40]'),
    ('P23', '[0.25, 0.25, 0.50]'),
    ('P24', '[0.20, 0.20, 0.60]'),
    ('P25', '[0.15, 0.15, 0.70]'),
    ('P26', '[0.10, 0.10, 0.80]'),
    ('P27', '[0.05, 0.05, 0.90]'),
    ('P28', '[0.02, 0.02, 0.95]'),
    ('P29', '[0.01, 0.01, 0.98]'),
    ('P30', '[0.00, 0.00, 1.00]'),
]

if not table_exists:
    cur.executemany(
        "INSERT INTO products_ex1 (name, embedding) VALUES (%s, %s)",
        data
    )
    conn.commit()
    print(f"Wstawiono {len(data)} produktów.")
else:
    print("Dane już istnieją — pomijam insert.")

Dane już istnieją — pomijam insert.


## Zapytanie a) — najbliższy do [0.9, 0.1, 0.0]

In [11]:
query_a = '[0.9, 0.1, 0.0]'

cur.execute("""
    SELECT name, embedding, embedding <=> %s::vector AS distance
    FROM products_ex1
    ORDER BY embedding <=> %s::vector
    LIMIT 1;
""", (query_a, query_a))

row = cur.fetchone()
print(f"Zapytanie: {query_a}")
print(f"Najbliższy produkt: {row[0]}")
print(f"Wektor:             {row[1]}")
print(f"Odległość kosinusowa: {row[2]:.6f}")

Zapytanie: [0.9, 0.1, 0.0]
Najbliższy produkt: P2
Wektor:             [0.88,0.1,0.01]
Odległość kosinusowa: 0.000067


## Zapytanie b) — najbliższy do [0.1, 0.9, 0.0]

In [12]:
query_b = '[0.1, 0.9, 0.0]'

cur.execute("""
    SELECT name, embedding, embedding <=> %s::vector AS distance
    FROM products_ex1
    ORDER BY embedding <=> %s::vector
    LIMIT 1;
""", (query_b, query_b))

row = cur.fetchone()
print(f"Zapytanie: {query_b}")
print(f"Najbliższy produkt: {row[0]}")
print(f"Wektor:             {row[1]}")
print(f"Odległość kosinusowa: {row[2]:.6f}")

Zapytanie: [0.1, 0.9, 0.0]
Najbliższy produkt: P12
Wektor:             [0.1,0.88,0.01]
Odległość kosinusowa: 0.000067


## Top 3 dla obu zapytań (porównanie)

In [13]:
for label, query_vec in [('a) [0.9, 0.1, 0.0]', '[0.9, 0.1, 0.0]'),
                          ('b) [0.1, 0.9, 0.0]', '[0.1, 0.9, 0.0]')]:
    cur.execute("""
        SELECT name, embedding, ROUND((embedding <=> %s::vector)::numeric, 6) AS distance
        FROM products_ex1
        ORDER BY embedding <=> %s::vector
        LIMIT 3;
    """, (query_vec, query_vec))

    rows = cur.fetchall()
    print(f"\nZapytanie {label}")
    print(f"{'Nazwa':<8} {'Wektor':<25} {'Odległość':>12}")
    print("-" * 50)
    for r in rows:
        print(f"{r[0]:<8} {str(r[1]):<25} {r[2]:>12}")


Zapytanie a) [0.9, 0.1, 0.0]
Nazwa    Wektor                       Odległość
--------------------------------------------------
P2       [0.88,0.1,0.01]               0.000067
P1       [0.91,0.05,0.02]              0.001795
P3       [0.85,0.15,0.05]              0.003718

Zapytanie b) [0.1, 0.9, 0.0]
Nazwa    Wektor                       Odległość
--------------------------------------------------
P12      [0.1,0.88,0.01]               0.000067
P11      [0.05,0.9,0.02]               0.001767
P13      [0.15,0.85,0.05]              0.003718


## Wyjaśnienie
--- 
Operator `<=>` mierzy **kąt między wektorami** — im mniejszy kąt, tym produkty są bardziej podobne.

Wektor `[0.9, 0.1, 0.0]` "patrzy" głównie w kierunku pierwszej osi, a `[0.1, 0.9, 0.0]` — drugiej. Każde zapytanie zwraca produkt, który leży najbliżej tego samego kierunku:

- `[0.9, 0.1, 0.0]` → **P2** `[0.88, 0.10, 0.01]` — duża pierwsza składowa
- `[0.1, 0.9, 0.0]` → **P12** `[0.10, 0.88, 0.01]` — duża druga składowa

Dwa różne kierunki w przestrzeni → dwa różne najbliższe produkty.

In [14]:
cur.close()
conn.close()